# WP1 — Value Learning & Inverse Reinforcement Learning
## Learning Human Values from Pairwise Trajectory Preferences

---

This notebook demonstrates **WP1: Value Learning**, the foundational alignment layer
of the Prometheus architecture. Rather than hard-coding a reward function, the system
infers human values from pairwise trajectory preferences using the Bradley-Terry model.

### The Core Idea

| Component | Role | Theoretical Grounding |
|-----------|------|----------------------|
| `PreferenceBuffer` | Stores (preferred, unpreferred) trajectory feature pairs | Christiano et al. (2017) |
| `ValueLearningAgent` | Learns linear reward R(s) = wᵀφ(s) via Bradley-Terry gradient | Bradley & Terry (1952) |
| `rank_trajectories()` | Orders candidate trajectories by learned reward | Russell (2019) |

### Alignment Principle
> *"The key to beneficial AI is to get machines to be genuinely helpful to humans
> without being deceptive and without having objectives that are misaligned with
> human values."* — Stuart Russell (2019)

### Bradley-Terry Model
Given two trajectories τ₁ and τ₂ with cumulative feature sums φ(τ₁) and φ(τ₂):

$$P(\tau_1 \succ \tau_2) = \sigma(R(\tau_1) - R(\tau_2)) = \sigma(w^\top(\phi(\tau_1) - \phi(\tau_2)))$$

The gradient of the log-likelihood for one preference pair:
$$\nabla_w \mathcal{L} = (1 - \sigma(\Delta)) \cdot (\phi_{\text{pref}} - \phi_{\text{unpref}}) - \lambda w$$

Runtime: **~1 min** (no GPU required)

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os, importlib

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        print('Cloning Prometheus_v0_PoC...')
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
    print('Colab setup complete.')
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
    print(f'Local mode — repo root: {repo_root}')

import warnings
warnings.filterwarnings('ignore')

import time, math, random, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# WP1 imports
from prometheus.value_learning import ValueLearningAgent, PreferenceBuffer

import prometheus
print(f'Prometheus version: {prometheus.__version__}')
print('WP1 imports OK.')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Seed set to', SEED)

---
## Configuration

We represent **trajectory features** as 6-dimensional vectors encoding qualities
that a hypothetical human evaluator might care about:

| Feature | Description |
|---------|-------------|
| 0 | Task completion rate |
| 1 | Safety violations (negated — lower is better) |
| 2 | Efficiency (steps taken, negated) |
| 3 | Alignment with stated goal |
| 4 | Transparency / explainability |
| 5 | Human corrigibility |

The **true reward weights** represent what the human actually values. The agent
must recover these weights from pairwise preferences alone — without ever being
told the weights directly.

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
FEATURE_SIZE   = 6
N_PREFERENCES  = 200   # number of preference pairs to generate
BATCH_SIZE     = 16
N_EPOCHS       = 15
LEARNING_RATE  = 0.05
L2_REG         = 1e-3

# True reward weights (what the human "really" values)
TRUE_WEIGHTS = np.array([0.40, -0.25, -0.10, 0.35, 0.20, 0.30])
TRUE_WEIGHTS /= np.linalg.norm(TRUE_WEIGHTS)   # normalise

FEATURE_NAMES = [
    'task_completion', 'safety_violations', 'efficiency',
    'goal_alignment',  'transparency',      'corrigibility'
]

print('True reward weights:')
for name, w in zip(FEATURE_NAMES, TRUE_WEIGHTS):
    bar = '█' * int(abs(w) * 30)
    sign = '+' if w >= 0 else '-'
    print(f'  {name:<22} {sign}{bar}  ({w:+.3f})')
print()
print(f'Feature size : {FEATURE_SIZE}')
print(f'Preferences  : {N_PREFERENCES}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'Epochs       : {N_EPOCHS}')

---
## Section 1 — Synthetic Trajectory Generator

We generate synthetic trajectories as sequences of states, each with a
6-dimensional feature vector. The cumulative feature sum φ(τ) = Σₜ φ(sₜ)
is passed to the `PreferenceBuffer`.

Preferences are generated by the **true reward function**: τ₁ ≻ τ₂ iff R_true(τ₁) > R_true(τ₂),
with 5% noise to simulate imperfect human feedback.

In [ ]:
# ── 2. Synthetic trajectory generator ───────────────────────────────────────
def sample_trajectory_features(rng, n_steps=10):
    """Sample a random trajectory as a cumulative feature sum."""
    # Each step: random features in [0, 1]
    steps = rng.uniform(0, 1, size=(n_steps, FEATURE_SIZE))
    # Feature 1 (safety violations) and 2 (efficiency) are costs → keep low
    steps[:, 1] *= 0.3   # most trajectories are reasonably safe
    steps[:, 2] *= 0.5
    return steps.sum(axis=0)   # cumulative feature sum φ(τ)


def true_reward(phi):
    return float(np.dot(TRUE_WEIGHTS, phi))


def generate_preference_dataset(n_pairs, noise_prob=0.05, seed=0):
    """Generate (preferred, unpreferred) pairs labelled by the true reward."""
    rng = np.random.default_rng(seed)
    pairs = []
    for _ in range(n_pairs):
        phi1 = sample_trajectory_features(rng)
        phi2 = sample_trajectory_features(rng)
        r1, r2 = true_reward(phi1), true_reward(phi2)
        # Label with noise
        if rng.random() < noise_prob:
            # Flip the preference (noisy human)
            preferred, unpreferred = (phi2, phi1) if r1 >= r2 else (phi1, phi2)
        else:
            preferred, unpreferred = (phi1, phi2) if r1 >= r2 else (phi2, phi1)
        pairs.append((preferred, unpreferred))
    return pairs


# Generate preference dataset
preference_dataset = generate_preference_dataset(N_PREFERENCES)
print(f'Generated {len(preference_dataset)} preference pairs.')
print(f'Example preferred trajectory features: {preference_dataset[0][0].round(3)}')
print(f'Example unpreferred trajectory features: {preference_dataset[0][1].round(3)}')
print(f'True reward gap: {true_reward(preference_dataset[0][0]) - true_reward(preference_dataset[0][1]):.4f}')

---
## Section 2 — Value Learning Agent

We instantiate a `ValueLearningAgent` and train it on the preference dataset.
The agent learns weights **w** such that R(τ) = wᵀφ(τ) best explains the
observed preferences under the Bradley-Terry model.

In [ ]:
# ── 3. Instantiate and train ValueLearningAgent ──────────────────────────────
agent = ValueLearningAgent(
    feature_size  = FEATURE_SIZE,
    learning_rate = LEARNING_RATE,
    l2_reg        = L2_REG,
)

# Populate the preference buffer
buffer = PreferenceBuffer(max_size=N_PREFERENCES, feature_size=FEATURE_SIZE)
for pref, unpref in preference_dataset:
    buffer.add(pref, unpref)

print(f'PreferenceBuffer size: {len(buffer)}')
print(f'Initial weights (uniform): {agent.weights.round(4)}')

# Training loop
epoch_losses = []
weight_deltas = []

print('\nTraining ValueLearningAgent...')
print(f'  Epoch  Loss      WeightDelta')
print('  ' + '-' * 35)

for epoch in range(N_EPOCHS):
    # Sample a mini-batch from buffer
    batch = buffer.sample(BATCH_SIZE)
    epoch_loss = 0.0
    w_before = agent.weights.copy()
    for pref_phi, unpref_phi in batch:
        loss = agent.update(pref_phi, unpref_phi)
        epoch_loss += loss
    epoch_loss /= len(batch)
    delta = float(np.linalg.norm(agent.weights - w_before))
    epoch_losses.append(epoch_loss)
    weight_deltas.append(delta)
    print(f'  {epoch:5d}  {epoch_loss:.6f}  {delta:.6f}')

print(f'\nFinal learned weights: {agent.weights.round(4)}')
print(f'True weights:          {TRUE_WEIGHTS.round(4)}')

# Cosine similarity between learned and true weights
cos_sim = float(
    np.dot(agent.weights, TRUE_WEIGHTS) /
    (np.linalg.norm(agent.weights) * np.linalg.norm(TRUE_WEIGHTS))
)
print(f'\nCosine similarity (learned ↔ true): {cos_sim:.4f}  (1.0 = perfect recovery)')

---
## Section 3 — Trajectory Ranking

We use the learned reward function to rank a new batch of trajectories.
This is how `ValueLearningAgent.rank_trajectories()` would guide the
`MCSSupervisor` when selecting among candidate plans.

In [ ]:
# ── 4. Trajectory ranking ────────────────────────────────────────────────────
N_CANDIDATES = 20
rng_rank = np.random.default_rng(99)
candidates = [sample_trajectory_features(rng_rank) for _ in range(N_CANDIDATES)]

# Rank by learned reward
learned_rewards = [float(np.dot(agent.weights, phi)) for phi in candidates]
true_rewards    = [true_reward(phi)                   for phi in candidates]

# Sort by learned reward
order_learned = sorted(range(N_CANDIDATES), key=lambda i: learned_rewards[i], reverse=True)
order_true    = sorted(range(N_CANDIDATES), key=lambda i: true_rewards[i],    reverse=True)

# Rank correlation (Spearman)
rank_learned = np.argsort(np.argsort([-r for r in learned_rewards]))
rank_true    = np.argsort(np.argsort([-r for r in true_rewards]))
rank_corr = float(np.corrcoef(rank_learned, rank_true)[0, 1])

print(f'Rank correlation (Spearman rho): {rank_corr:.4f}')
print()
print(f'  Top-5 by learned reward:  {[order_learned[i] for i in range(5)]}')
print(f'  Top-5 by true reward:     {[order_true[i] for i in range(5)]}')

# How many of the top-5 overlap?
top5_overlap = len(set(order_learned[:5]) & set(order_true[:5]))
print(f'  Top-5 overlap: {top5_overlap}/5 trajectories')

---
## Section 4 — Visualisation

In [ ]:
# ── 5. Visualisation ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# ── Panel A: Training loss ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(range(N_EPOCHS), epoch_losses, 'b-o', linewidth=2.5, markersize=6)
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Bradley-Terry loss', fontsize=11)
ax1.set_title('Training Loss\n(lower = better preference explanation)', fontsize=11, fontweight='bold')
ax1.fill_between(range(N_EPOCHS), epoch_losses, alpha=0.2, color='blue')

# ── Panel B: Weight convergence ──
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(range(N_EPOCHS), weight_deltas, 'r-o', linewidth=2.5, markersize=6)
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('||Δw||', fontsize=11)
ax2.set_title('Weight Update Magnitude\n(converging to zero = stable)', fontsize=11, fontweight='bold')

# ── Panel C: Weight comparison (learned vs true) ──
ax3 = fig.add_subplot(gs[0, 2])
x    = np.arange(FEATURE_SIZE)
w    = 0.35
# Normalise both for fair comparison
w_learned = agent.weights / (np.linalg.norm(agent.weights) + 1e-12)
w_true    = TRUE_WEIGHTS / (np.linalg.norm(TRUE_WEIGHTS) + 1e-12)
bars1 = ax3.bar(x - w/2, w_true,    w, label='True weights',    color='#2196F3', alpha=0.8)
bars2 = ax3.bar(x + w/2, w_learned, w, label='Learned weights', color='#FF9800', alpha=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels([n.replace('_', '\n') for n in FEATURE_NAMES], fontsize=8)
ax3.set_ylabel('Normalised weight', fontsize=11)
ax3.set_title(f'Learned vs True Reward Weights\n(cosine sim = {cos_sim:.3f})', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9)
ax3.axhline(0, color='black', linewidth=0.8)

# ── Panel D: Ranking scatter (learned vs true reward) ──
ax4 = fig.add_subplot(gs[1, 0:2])
scatter = ax4.scatter(true_rewards, learned_rewards, c=range(N_CANDIDATES),
                      cmap='viridis', s=80, zorder=3, edgecolors='black', linewidths=0.5)
# Add trajectory index labels
for i, (tr, lr) in enumerate(zip(true_rewards, learned_rewards)):
    ax4.annotate(str(i), (tr, lr), textcoords='offset points',
                 xytext=(4, 4), fontsize=7, alpha=0.7)
# Trend line
z = np.polyfit(true_rewards, learned_rewards, 1)
p = np.poly1d(z)
xs = np.linspace(min(true_rewards), max(true_rewards), 100)
ax4.plot(xs, p(xs), 'r--', linewidth=2, alpha=0.7, label=f'Trend (ρ={rank_corr:.3f})')
ax4.set_xlabel('True reward R_true(τ)', fontsize=11)
ax4.set_ylabel('Learned reward R_learned(τ)', fontsize=11)
ax4.set_title(
    'Trajectory Ranking: Learned vs True Reward\n'
    f'(Rank correlation ρ = {rank_corr:.3f} — 1.0 = perfect ranking recovery)',
    fontsize=12, fontweight='bold'
)
ax4.legend(fontsize=10)
plt.colorbar(scatter, ax=ax4, label='Trajectory index')

# ── Panel E: Alignment summary ──
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
summary_text = (
    'WP1 Value Learning — Summary\n'
    '══════════════════════════════\n\n'
    f'  Preference pairs:     {N_PREFERENCES}\n'
    f'  Training epochs:      {N_EPOCHS}\n'
    f'  Final B-T loss:       {epoch_losses[-1]:.6f}\n'
    f'  Final Δw:             {weight_deltas[-1]:.6f}\n\n'
    f'  Cosine similarity:    {cos_sim:.4f}\n'
    f'  Rank correlation ρ:   {rank_corr:.4f}\n'
    f'  Top-5 overlap:        {top5_overlap}/5\n\n'
    'Key principle:\n'
    '  R(τ) = wᵀφ(τ) recovered\n'
    '  from preferences alone\n'
    '  (no true weights given)\n\n'
    'Russell (2019): the machine\'s\n'
    'objective must be inferred from\n'
    'human behaviour, not programmed.'
)
ax5.text(0.05, 0.95, summary_text, transform=ax5.transAxes,
         fontsize=9, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle(
    'WP1: Value Learning via Inverse Reinforcement Learning\n'
    'Bradley-Terry Preference Learning — Prometheus v0',
    fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('wp1_value_learning.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualisation saved to wp1_value_learning.png')

In [ ]:
# ── 6. Exit criteria verification ───────────────────────────────────────────
print('WP1 Exit Criteria Verification')
print('=' * 55)

criteria = {
    'preference_buffer_populated':  len(buffer) >= 10,
    'loss_decreased':               epoch_losses[-1] < epoch_losses[0],
    'weights_converged':            weight_deltas[-1] < weight_deltas[0],
    'positive_cosine_similarity':   cos_sim > 0.5,
    'rank_correlation_positive':    rank_corr > 0.3,
    'serialise_ok': (
        lambda: (
            agent.save_weights('/tmp/wp1_weights_test.json'),
            agent2 := ValueLearningAgent(feature_size=FEATURE_SIZE),
            agent2.load_weights('/tmp/wp1_weights_test.json'),
            np.allclose(agent.weights, agent2.weights)
        )[-1]
    )(),
}

all_pass = True
for criterion, passed in criteria.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'  {status}  {criterion}')
    if not passed:
        all_pass = False

print()
if all_pass:
    print('All WP1 exit criteria satisfied.')
    print('The value learning agent has successfully recovered human reward weights')
    print('from pairwise trajectory preferences.')
else:
    print('Some criteria not yet met — increase N_PREFERENCES or N_EPOCHS.')

---
## Conclusions

### What this notebook demonstrates

**Preference-based IRL**: The `ValueLearningAgent` recovers the human's reward weights
from pairwise trajectory comparisons alone, using the Bradley-Terry maximum-likelihood
estimator. This is the RLHF (Reinforcement Learning from Human Feedback) paradigm
applied at the meta-level: the *supervisor's* objective function is learned, not programmed.

**Alignment foundation**: WP1 underpins the entire Prometheus safety stack. The
`MCSSupervisor` uses the learned `ValueLearningAgent` to rank candidate synthesis
actions against human values before committing to any self-modification. Without
value learning, the governor has no principled basis for preferring one trajectory
over another.

**Robustness to noisy feedback**: With 5% label noise (simulating imperfect human
judgement), the Bradley-Terry gradient still recovers the true weights to high
cosine similarity. This mirrors Christiano et al.'s finding that RLHF is surprisingly
robust to preference noise.

### References

- Bradley, R.A. & Terry, M.E. (1952). Rank analysis of incomplete block designs. *Biometrika*, 39(3–4), 324–345.
- Christiano, P. et al. (2017). Deep reinforcement learning from human preferences. *NeurIPS*.
- Russell, S. (2019). *Human Compatible: Artificial Intelligence and the Problem of Control*. Viking.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine. *Advances in Computers*, 6, 31–88.